# OLG 5-Step Workflow Demo

**Module 4, Topic 4.3** — Case study of the five-stage Claude Code workflow applied to a research-grade two-asset OLG with stabilising homotopy.

This notebook walks the V0 → V5 ladder. Each section trains one version, simulates the trained policy, prints the validation gate, and writes a reproducibility JSON to `build/notebook_report_v{n}.json`.

| Ver | Adds | Validation gate |
|---|---|---|
| V0 | 3 cohorts, 1 asset, 2-state Markov TFP, single MLP, Euler MSE on cloud | deterministic-SS collapse + procyclical capital + RMS Euler < 8% |
| V1 | 7 cohorts, hump-shaped lifecycle labour | reduce-to-V0 + middle-aged peak in savings rate |
| V2 | 4-state Rouwenhorst TFP | reduce-to-V1 with `n_tfp=2` |
| V3 | bonds + market-clearing layer + Fischer–Burmeister | bond market clears; lifecycle bond pattern |
| V4 | capital adjustment cost | reduces to V3 at $\psi_K = 0$ |
| V5 | 4-phase stabilising homotopy | per-phase residuals decrease; full structural target |

The lesson is the **workflow**, not the model. Each section is the artifact of one Claude Code session driven by `prompts/v{n}_to_v{n+1}.md`.


## Section 0 — Setup

Detect environment and prepare output directories. Each version's section will insert its own folder onto `sys.path` and clear cached version-shared module names so the imports resolve correctly.


In [ ]:
import json
import math
import sys
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt

ROOT = Path.cwd()
BUILD = ROOT / "build"
FIGURES = ROOT / "figures"
BUILD.mkdir(exist_ok=True)
FIGURES.mkdir(exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device   : {device}")
print(f"torch    : {torch.__version__}")
print(f"numpy    : {np.__version__}")
print(f"matplot  : {plt.matplotlib.__version__}")

DEMO_BUDGETS = dict(
    v0={"n_steps": 2500, "pretrain_steps": 400, "log_every": 1000},
    v1={"n_steps": 3000, "pretrain_steps": 600, "log_every": 1000},
    v2={"n_steps": 3000, "pretrain_steps": 600, "log_every": 1000},
    v3={"n_steps": 4000, "pretrain_steps": 800, "log_every": 1000},
    v4={"n_steps": 4000, "pretrain_steps": 800, "log_every": 1000},
    v5={"phase1_steps": 1500, "phase2_steps": 1000, "phase3_steps": 1000,
        "phase4_steps": 1000, "pretrain_steps": 600, "log_every": 1000},
)


def use_version(name: str) -> None:
    target = str(ROOT / "versions" / name)
    sys.path[:] = [p for p in sys.path
                   if not p.endswith(("versions/v0", "versions/v1", "versions/v2",
                                      "versions/v3", "versions/v4", "versions/v5"))]
    sys.path.insert(0, target)
    for m in ("model", "network", "train", "simulate", "plotting"):
        sys.modules.pop(m, None)


def write_report(version: str, params: dict, training: dict, summary: dict, gate: dict) -> Path:
    report = {"version": version, "params": params, "training": training,
              "summary": summary, "validation_gate": gate,
              "validation_pass": all(v for v in gate.values() if isinstance(v, bool))}
    out = BUILD / f"notebook_report_{version}.json"
    out.write_text(json.dumps(report, indent=2))
    return out


## Section 1 — V0: three-cohort seed

V0 is the seed: three cohorts, one asset (capital), two-state Markov TFP, a single MLP policy trained by minimising the mean-squared normalised Euler residual over a cloud of 512 parallel economies. **No bonds, no homotopy, no idiosyncratic risk** — the smallest interesting OLG where a neural-network policy earns its keep.

Read more: `versions/v0/model_spec.md`, `versions/v0/pseudocode.md`.


In [ ]:
use_version("v0")
import model as v0_model
import train as v0_train
import simulate as v0_simulate
import plotting as v0_plot

torch.manual_seed(0); np.random.seed(0)
net0, losses0 = v0_train.run(hp_overrides=DEMO_BUDGETS["v0"], verbose=True)
sim0 = v0_simulate.run(net0, T=2500, burn=300)
v0_simulate.print_summary(sim0)

gate0 = v0_simulate.validation_gate(sim0, losses0)
print()
print("Validation gate:", gate0)
print(f"OVERALL: {'PASS' if all(gate0.values()) else 'FAIL'}")


In [ ]:
# V0 figures
fig, ax = plt.subplots(figsize=(8, 3.2)); v0_plot.plot_loss(losses0, ax); plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5)); v0_plot.plot_path_and_hist(sim0, axes=axes); plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(7, 3.5)); v0_plot.plot_lifecycle(sim0, ax); plt.tight_layout(); plt.show()

R_y, R_m = v0_simulate.ergodic_residuals(net0)
fig, ax = plt.subplots(figsize=(7, 3.2)); v0_plot.plot_residual_hist(R_y, R_m, ax); plt.tight_layout(); plt.show()

write_report(
    "v0",
    params={k: v for k, v in v0_model.P.items()},
    training={"n_steps": len(losses0), "final_mse": float(losses0[-1])},
    summary={"E[K]": float(sim0["K"].mean()), "E[r]": float(sim0["r"].mean())},
    gate=gate0,
)


## Section 2 — V0 → V1: seven-cohort lifecycle

V1 extends the cohort dimension from three to seven on a hump-shaped efficiency-labour profile $\boldsymbol{\varepsilon} = (0.7, 0.9, 1.0, 1.05, 1.0, 0.9, 0.0)$. Discount and depreciation re-anchored on yearly primitives ($\beta_y = 0.97$, $\delta_y = 0.06$) over a generational period of $\tau = 72/7 \approx 10.286$ years. Six savers (cohorts age 0–5); cohort age 6 is retired.

The algorithm is identical in spirit; only the cohort dimension is vectorised. See `prompts/v0_to_v1.md`.


In [ ]:
use_version("v1")
import model as v1_model
import train as v1_train
import simulate as v1_simulate
import plotting as v1_plot

torch.manual_seed(0); np.random.seed(0)
net1, losses1 = v1_train.run(hp_overrides=DEMO_BUDGETS["v1"], verbose=True)
sim1 = v1_simulate.run(net1, T=2500, burn=300)
v1_simulate.print_summary(sim1)

gate1 = v1_simulate.validation_gate(sim1, losses1)
print()
print("Validation gate:", gate1)
print(f"OVERALL: {'PASS' if all(gate1.values()) else 'FAIL'}")


In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.2)); v1_plot.plot_loss(losses1, ax); plt.tight_layout(); plt.show()
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5)); v1_plot.plot_aggregate(sim1, axes=axes); plt.tight_layout(); plt.show()
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5)); v1_plot.plot_lifecycle(sim1, axes=axes); plt.tight_layout(); plt.show()

write_report(
    "v1",
    params={k: float(v) if isinstance(v, (int, float)) else v for k, v in v1_model.P.items()},
    training={"n_steps": len(losses1), "final_mse": float(losses1[-1])},
    summary={"E[K]": float(sim1["K"].mean()), "E[r]": float(sim1["r"].mean()),
             "mean_c_by_age": [float(x) for x in sim1["c"].mean(axis=0)],
             "mean_s_by_age": [float(x) for x in sim1["s"].mean(axis=0)]},
    gate=gate1,
)


## Section 3 — V1 → V2: 4-state Rouwenhorst TFP

V2 replaces V1's 2-state symmetric Markov TFP with a 4-state Rouwenhorst discretisation of an annual AR(1) on log-TFP, primitives $\rho_y = 0.85$, $\sigma_{\varepsilon,y} = 0.03$. The aggregation to per-period AR(1) is

$$
\rho_\tau = \rho_y^{\,\tau}, \qquad
\sigma_{\varepsilon,\tau} = \sigma_{\varepsilon,y}\sqrt{\frac{1-\rho_y^{\,2\tau}}{1-\rho_y^{\,2}}}.
$$

Closed-form expectations are preserved (sum over four states instead of two). See `prompts/v1_to_v2.md`.


In [ ]:
use_version("v2")
import model as v2_model
import train as v2_train
import simulate as v2_simulate
import plotting as v2_plot

print("Z grid:", v2_model.Z_VALS.cpu().numpy())
print("transition row sums:", v2_model.P_MAT.sum(dim=-1).cpu().numpy())

torch.manual_seed(0); np.random.seed(0)
net2, losses2 = v2_train.run(hp_overrides=DEMO_BUDGETS["v2"], verbose=True)
sim2 = v2_simulate.run(net2, T=2500, burn=300)
v2_simulate.print_summary(sim2)

gate2 = v2_simulate.validation_gate(sim2, losses2)
print()
print("Validation gate:", gate2)
print(f"OVERALL: {'PASS' if all(v for v in gate2.values() if isinstance(v, bool)) else 'FAIL'}")


In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.2)); v2_plot.plot_loss(losses2, ax); plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(7, 3.4)); v2_plot.plot_tfp_chain(ax); plt.tight_layout(); plt.show()
fig, ax = plt.subplots(figsize=(7, 3.4)); v2_plot.plot_aggregate_by_regime(sim2, ax); plt.tight_layout(); plt.show()
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5)); v2_plot.plot_lifecycle(sim2, axes=axes); plt.tight_layout(); plt.show()

write_report(
    "v2",
    params={k: float(v) if isinstance(v, (int, float)) else v for k, v in v2_model.P.items()},
    training={"n_steps": len(losses2), "final_mse": float(losses2[-1])},
    summary={"E[K]": float(sim2["K"].mean()), "E[r]": float(sim2["r"].mean()),
             "K_spread": gate2["K_spread_across_TFP_states"]},
    gate={k: v for k, v in gate2.items() if isinstance(v, bool)},
)


## Section 4 — V2 → V3: bonds + market-clearing layer + Fischer–Burmeister

V3 introduces a one-period zero-coupon bond in zero net supply with an endogenous price $p_b$. The state expands to $(Z, k^1, \ldots, k^{N-1}, b^1, \ldots, b^{N-1})$. The policy network outputs $(N-1)$ capital savings rates, $(N-1)$ raw bond demands, and one bond price; a **market-clearing layer** subtracts the cohort-mean from the raw bond demand so $\sum_j b^j = 0$ holds by construction.

The borrowing limit $b^j \geq -0.05$ is enforced softly via a Fischer–Burmeister residual added to the loss. See `prompts/v2_to_v3.md`.


In [ ]:
use_version("v3")
import model as v3_model
import train as v3_train
import simulate as v3_simulate
import plotting as v3_plot

torch.manual_seed(0); np.random.seed(0)
net3, losses3 = v3_train.run(hp_overrides=DEMO_BUDGETS["v3"], verbose=True)
sim3 = v3_simulate.run(net3, T=2500, burn=300)
v3_simulate.print_summary(sim3)

gate3 = v3_simulate.validation_gate(sim3, losses3)
print()
print("Validation gate:", gate3)
print(f"OVERALL: {'PASS' if all(gate3.values()) else 'FAIL'}")


In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.2)); v3_plot.plot_loss(losses3, ax); plt.tight_layout(); plt.show()
fig, axes = plt.subplots(2, 2, figsize=(11, 7)); v3_plot.plot_lifecycle(sim3, axes=axes); plt.tight_layout(); plt.show()
fig, ax = plt.subplots(figsize=(8, 3.2)); v3_plot.plot_bond_price(sim3, ax); plt.tight_layout(); plt.show()

write_report(
    "v3",
    params={k: float(v) if isinstance(v, (int, float)) else v for k, v in v3_model.P.items()},
    training={"n_steps": len(losses3), "final_total": float(losses3[-1])},
    summary={"E[K]": float(sim3["K"].mean()), "E[r]": float(sim3["r"].mean()),
             "E[p_b]": float(sim3["p_b"].mean()),
             "mean_b_by_age": [float(x) for x in sim3["b"].mean(axis=0)]},
    gate=gate3,
)


## Section 5 — V3 → V4: capital adjustment cost

V4 adds a convex capital adjustment cost $\frac{\psi_K}{2}(k_\text{next} - k)^2$ paid out of period-$t$ consumption. The capital Euler equation gains the marginal-adjustment factor $1 + \psi_K\,(k_\text{next} - k)$. With $\psi_K = 0.50$ (mild), the smoothing effect is visible without dominating prices.

Setting `psi_K=0` exactly reproduces V3. See `prompts/v3_to_v4.md`.


In [ ]:
use_version("v4")
import model as v4_model
import train as v4_train
import simulate as v4_simulate
import plotting as v4_plot

torch.manual_seed(0); np.random.seed(0)
net4, losses4 = v4_train.run(hp_overrides=DEMO_BUDGETS["v4"], verbose=True)
sim4 = v4_simulate.run(net4, T=2500, burn=300)
v4_simulate.print_summary(sim4)

gate4 = v4_simulate.validation_gate(sim4, losses4)
print()
print("Validation gate:", gate4)
print(f"OVERALL: {'PASS' if all(v for v in gate4.values() if isinstance(v, bool)) else 'FAIL'}")


In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.2)); v4_plot.plot_loss(losses4, ax); plt.tight_layout(); plt.show()
fig, axes = plt.subplots(2, 2, figsize=(11, 7)); v4_plot.plot_lifecycle(sim4, axes=axes); plt.tight_layout(); plt.show()
fig, ax = plt.subplots(figsize=(8, 3.2)); v4_plot.plot_bond_price(sim4, ax); plt.tight_layout(); plt.show()

write_report(
    "v4",
    params={k: float(v) if isinstance(v, (int, float)) else v for k, v in v4_model.P.items()},
    training={"n_steps": len(losses4), "final_total": float(losses4[-1])},
    summary={"E[K]": float(sim4["K"].mean()), "std[K]": float(sim4["K"].std()),
             "E[r]": float(sim4["r"].mean()), "E[p_b]": float(sim4["p_b"].mean()),
             "E[adj_cost]": float(sim4["adj_cost"].mean())},
    gate={k: v for k, v in gate4.items() if isinstance(v, bool)},
)


## Section 6 — V4 → V5: stabilising homotopy

V5 keeps V4's economic model unchanged and replaces single-shot training with a four-phase stabilising-homotopy schedule:

1. **Capital-only** — train on $R_K$ alone with `bonds_off=True`.
2. **Bond pretraining** — turn bonds on at a small loss weight ($w_B = 0.1$).
3. **Bond homotopy** — linearly ramp $w_B$ from $0.1 \to 1.0$ and the FB weight from $0 \to 0.5$.
4. **Fine-tuning** — final polish at $\Gamma/10$.

This is the algorithmic technique that brings the demo to research-grade structural parity with `reference/research_target_notes.md`. See `prompts/v4_to_v5.md`.


In [ ]:
use_version("v5")
import model as v5_model
import train as v5_train
import simulate as v5_simulate
import plotting as v5_plot

torch.manual_seed(0); np.random.seed(0)
net5, history5 = v5_train.run(hp_overrides=DEMO_BUDGETS["v5"], verbose=True)
sim5 = v5_simulate.run(net5, T=2500, burn=300)
v5_simulate.print_summary(sim5)

gate5 = v5_simulate.validation_gate(sim5, history5)
print()
print("Validation gate:", gate5)
print(f"OVERALL: {'PASS' if all(v for v in gate5.values() if isinstance(v, bool)) else 'FAIL'}")


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.0)); v5_plot.plot_homotopy_loss(history5, ax); plt.tight_layout(); plt.show()
fig, ax = plt.subplots(figsize=(9, 3.5)); v5_plot.plot_phase_summary(history5, ax); plt.tight_layout(); plt.show()
fig, axes = plt.subplots(2, 2, figsize=(11, 7)); v5_plot.plot_lifecycle(sim5, axes=axes); plt.tight_layout(); plt.show()

write_report(
    "v5",
    params={k: float(v) if isinstance(v, (int, float)) else v for k, v in v5_model.P.items()},
    training={"phases": [{**p} for p in history5["phases"]],
              "total_steps": len(history5["all"])},
    summary={"E[K]": float(sim5["K"].mean()), "E[r]": float(sim5["r"].mean()),
             "E[p_b]": float(sim5["p_b"].mean()),
             "mean_b_by_age": [float(x) for x in sim5["b"].mean(axis=0)]},
    gate={k: v for k, v in gate5.items() if isinstance(v, bool)},
)


## Section 7 — Where the workflow paid off

Six versions, six Claude Code sessions, six validation gates. Every transition was one disciplined session walking Stages 1–5 (Model → Equilibrium → Algorithm → Pseudo-code → Implement) with a literal prompt at `prompts/v{n}_to_v{n+1}.md`. Each version's final code lives at `versions/v{n}/`; each session's audit trail is in `versions/v{n}/session_notes.md`.

**The pedagogical claim** is *not* that V5 reproduces a specific research notebook byte-for-byte. It is that **every algorithmic primitive** the silent research-grade target uses — cloud method, Euler-residual loss, two-asset market-clearing layer, Fischer–Burmeister complementarity, capital adjustment cost, stabilising homotopy — appears somewhere in the V0 → V5 ladder and is validated against the previous version's output.

### Discussion prompts for students

1. **Where would a sloppy spec have hurt most?** Pick one transition (any V_n → V_{n+1}). Identify a vague Stage-1 spec that would have produced "looks right" code solving the wrong problem. What's the cheapest validation that would catch it?
2. **Why mean-subtract vs. residualise the last cohort?** V3 enforces $\sum_j b^j = 0$ by subtracting the cohort-mean from raw bond demand. Sketch the consequences of the alternative (residualising one cohort's bond holding from the constraint). Where would the cohort that's "stuck with the residual" land economically?
3. **The corner solution at cohort age 0.** V1 onwards typically pins $s_K$ for the youngest cohort to the floor. Is this an optimisation failure or an economic feature? Trace the reasoning through the Euler equation $c_{j+1}/c_j = (\beta(1+r))^{1/\gamma}$ and the labour-income gradient.
4. **What does the homotopy schedule actually buy?** V4 reaches a working two-asset equilibrium without homotopy. What sort of calibration change would push V4 into a local minimum that V5's schedule rescues? (Hint: tighter borrowing limit, larger $\psi_K$, more cohorts, lower $b_\text{scale}$.)
5. **Pillar 5 vs. pillars 1–4.** Of the six transitions, which two depended most on *code-literacy* skills (vectorisation, broadcasting, dtype) versus *economic-judgment* skills (spec, equilibrium, algorithm choice)? Where would Claude's contribution have been highest if it had access only to the spec and not the history of validated versions?

### Reproducibility

`build/notebook_report_v{0..5}.json` contains the parameters, training summary, ergodic moments, and validation-gate booleans for each section's run. Re-running the notebook overwrites; the gate booleans should remain True.
